In [ ]:
# === QuakeMLab sample configuration (auto-added) ===
# By default the workflow runs on the bundled 5-event sample in data/sample/.
# To use your own archive, set QUAKEMLAB_SAMPLE or edit SFILE_PATH/EVENTS_DIR.
import os
SAMPLE = os.environ.get("QUAKEMLAB_SAMPLE") or next(
    (p for p in ("../data/sample", "data/sample") if os.path.isdir(p)), "data/sample")
SFILE_PATH = os.path.join(SAMPLE, "bulletin.out")      # Nordic (SEISAN) S-file
EXCEL_PATH = os.path.join(SAMPLE, "event_list.xlsx")   # optional AFAD list (absent for sample)
EVENTS_DIR = os.path.join(SAMPLE, "events")            # one folder per event, *.mseed (UTC)
DEMO_MODE  = True                                       # M4: run a single lightweight model
def models_to_run(full_list):
    return ["PhaseNetLight"] if DEMO_MODE else full_list
print("SAMPLE =", SAMPLE, "| exists:", os.path.isdir(SAMPLE))


# Manuel Picks

In [ ]:
from datetime import datetime, timedelta
from obspy import UTCDateTime
import os
import obspy
import plotly.graph_objects as go
from plotly.subplots import make_subplots

from obspy import UTCDateTime
import pandas as pd

from datetime import datetime

def safe_parse_pick_time(pick):
    """
    Pick zamanını güvenli biçimde UTCDateTime objesine çevirir.
    Hem ISO formatını hem 'HHMM SS.ss' tipindeki kısa formatları destekler.
    """
    try:
        # ISO tipi zamanları direkt döndür
        return UTCDateTime(pick["time"])
    except Exception:
        raw = pick.get("raw_time_str", "").strip()
        if not raw:
            raise TypeError("raw_time_str boş")

        try:
            # '1046 36.03' → 10:46:36.03
            hour = int(raw[:2])
            minute = int(raw[2:4])
            second = float(raw.split()[1])

            # tarih kısmı mevcutsa kullan
            base_date = pick["time"].split(" ")[0]
            # string oluştur
            full_str = f"{base_date} {hour:02d}:{minute:02d}:{second:06.3f}"
            return UTCDateTime(full_str)
        except Exception as e:
            raise TypeError(f"Pick parse hatası: {e}")


def plot_event_with_plotly(event_data, mseed_root_dir, 
                           output_filename="event_plot.html", 
                           station_to_plot=None):
    event_folder = event_data['event_folder_name']
    event_picks = event_data.get('picks', [])
    event_origin_time = UTCDateTime(event_data['origin_time'])
    
    folder_path = os.path.join(mseed_root_dir, event_folder)
    if not os.path.exists(folder_path):
        print(f"❌ '{event_folder}' klasörü bulunamadı.")
        return

    print(f"\n📂 Event Yükleniyor: {event_folder} ({event_data.get('location')})")

    try:
        st = obspy.read(os.path.join(folder_path, "*.mseed"))
        if len(st) == 0:
            print(f"   Klasörde .mseed dosyası bulunamadı: {folder_path}")
            return
    except Exception as e:
        print(f"   MSeed okuma hatası: {e}")
        return

    try:
        st.merge(method=1, fill_value=0)
        st.detrend("demean")
        st.filter("bandpass", freqmin=1.0, freqmax=45.0)
    except Exception as e:
        print(f"   Ön işleme (merge/filter) hatası: {e}")

    print(f"   Toplam {len(st)} trace (kanal) yüklendi ve işlendi.")

    if station_to_plot:
        print(f"   İstasyona göre filtreleniyor: {station_to_plot}")
        st = st.select(station=station_to_plot)
        if len(st) == 0:
            print(f"   ❌ UYARI: '{station_to_plot}' istasyonu bu event'in mseed verisinde bulunamadı.")
            return
        print(f"   Filtreleme sonrası {len(st)} trace (kanal) kaldı.")

    n_traces = len(st)
    trace_ids = [tr.id for tr in st]
    if n_traces == 0:
        print(f"   Görselleştirilecek trace bulunamadı.")
        return

    if n_traces > 1:
        max_allowed_spacing = 0.9 / (n_traces - 1)
        v_spacing = min(0.02, max_allowed_spacing)
    else:
        v_spacing = 0.02

    fig = make_subplots(rows=n_traces, cols=1, shared_xaxes=True, vertical_spacing=v_spacing, subplot_titles=trace_ids)
    print("   Plotly alt grafikleri oluşturuluyor...")

    # add traces
    for i, tr in enumerate(st, start=1):
        time_vector = tr.times("utcdatetime")  # list of python datetimes
        fig.add_trace(go.Scatter(x=time_vector, y=tr.data, mode='lines', name=tr.id), row=i, col=1)

    # pick'leri ekle
    print("   Pick'ler filtreleniyor ve grafiğe ekleniyor...")
    picks_added = 0
    for pick in event_picks:
        if station_to_plot and pick['station'] != station_to_plot:
            continue
        try:
            pick_time = safe_parse_pick_time(pick).datetime
            phase = pick["phase"].upper()
            color = "red" if phase == "P" else "blue"

            # 🔹 Dikey çizgi (fazlara göre renkli)
            fig.add_vline(
                x=pick_time,
                line_width=1,
                line_dash="dash",
                line_color=color,
            )

            # 🔹 Faz ismini üst kısımda farklı yüksekliklerde yaz
            y_pos = 0.97 if phase == "P" else 0.93  # P üstte, S altta olacak

            fig.add_annotation(
                text=f"{pick['phase']} @ {pick_time.strftime('%H:%M:%S.%f')[:-3]}",
                x=pick_time,
                yref="paper", y=y_pos,
                showarrow=False,
                font=dict(color=color, size=10),
                xanchor="center"
            )

            print(f"✅ {pick['station']} - {phase} çizildi @ {pick_time}")

        except Exception as e:
            print(f"⚠️ Pick eklenemedi: {pick} - Hata: {e}")
        picks_added+=1



    print(f"   {picks_added} adet manuel pick grafiğe eklendi.")

    plot_title = f"Event ID: {event_data.get('event_id','?')} - {event_origin_time.strftime('%Y-%m-%d %H:%M:%S')}"
    if station_to_plot:
        plot_title += f" (İstasyon: {station_to_plot})"
    fig.update_layout(title_text=plot_title, height=max(600, 150 * n_traces), showlegend=False, xaxis_title="Zaman (UTC)")

    # ensure x-range uses datetimes (not ISO strings) and includes picks
    start_range_dt = (event_origin_time - timedelta(seconds=5)).datetime
    end_range_dt   = (event_origin_time + timedelta(seconds=40)).datetime
    fig.update_xaxes(range=[start_range_dt, end_range_dt])

    fig.write_html(output_filename)
    print(f"\n✅ İnteraktif grafik başarıyla kaydedildi: {output_filename}")


# 1. Katalog (Ground Truth) dosyamız
json_path = "sfile_parsed.json" 

# 2. Ham MSEED dosyalarının bulunduğu ana klasör
mseed_root_dir = os.path.join(os.getcwd(), "Veri Seti_Turkiye")

# --- Verileri Yükle ---
df_events = pd.read_json(json_path)
    
# Klasör isimlerinden event_id -> folder_name mapping oluştur
folder_id_to_name = {}
if os.path.exists(mseed_root_dir):
    for folder_name in os.listdir(mseed_root_dir):
        if os.path.isdir(os.path.join(mseed_root_dir, folder_name)):
            try:
                event_id_str = folder_name.split('_')[0]
                folder_id_to_name[event_id_str] = folder_name
            except:
                pass

# df_events'e event_folder_name ekle
df_events['event_folder_name'] = df_events['event_id'].astype(str).map(folder_id_to_name)

df_events = df_events.dropna(subset=['event_folder_name'])

print(f"\nToplam {len(df_events)} adet mseed klasörü ile eşleşen event bulundu.")

# DataFrame'in ilk birkaç satırını göster
display(df_events[['origin_time', 'event_id', 'location', 'event_folder_name']].head())

## Dışarı aktar

In [ ]:
import pandas as pd
from obspy import UTCDateTime

def export_all_manual_picks(df_events, output_file="manual_picks.xlsx"):
    """
    df_events içindeki tüm manuel pickleri tek bir DataFrame'e çevirir
    ve event_id sütununu sol tarafa ekler.
    """
    all_data = []

    for idx, row in df_events.iterrows():
        event_id = row['event_id']
        manual_picks = row['picks']  # listenin kendisi

        for pick in manual_picks:
            pick_time = pick['time']
            # Eğer UTCDateTime ise datetime objesine çevir
            if hasattr(pick_time, 'datetime'):
                pick_time = pick_time.datetime
            all_data.append({
                'event_id': event_id,
                'station': pick['station'],
                'component': pick.get('component', ''),
                'phase': pick['phase'],
                'manuel_pick': pick_time
            })

    df_all = pd.DataFrame(all_data)
    df_all = df_all[['event_id', 'station', 'component', 'phase', 'manuel_pick']]

    df_all.to_excel(output_file, index=False)
    print(f"✅ Manuel pickler kaydedildi: {output_file}")
    return df_all



In [ ]:
# === run export on the parsed events (auto-added) ===
import pandas as pd
df_events = pd.read_json("sfile_parsed.json")
export_all_manual_picks(df_events, "manual_picks.xlsx")
print("manual_picks.xlsx written:", len(df_events), "events")
